# 09 · Inspect the project database

Run the first Airflow DAG before querying the tables.

## Imports

In [1]:
import os
import subprocess
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text

load_dotenv(Path('../.env'))
if not os.getenv('APP_DB_HOST'):
    kubeconfig = os.getenv('KUBECONFIG') or str(Path.home() / '.kube/project-3-k3s.yaml')
    os.environ['APP_DB_HOST'] = subprocess.check_output([
        'kubectl', '--kubeconfig', kubeconfig, '--request-timeout=10s', '-n', 'project-3',
        'get', 'svc', 'property-postgres', '-o', 'jsonpath={.spec.clusterIP}',
    ], text=True).strip()

sys.path.insert(0, str(Path('../src').resolve()))
from property_ml.settings import engine

db = engine()

## Batches

In [2]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM raw.batches ORDER BY group_number, batch_number'), conn))


,group_number,batch_number,run_id,received_at,row_count,status,received_columns,valid_rows,rejected_rows,duplicate_rows,quality
0,4,0,manual__2026-11-27T05:18:08.604029+00:00,2026-11-27 05:17:39.091793+00:00,73784,cleaned,"[acre_lot, bath, bed, brokered_by, city, house...",73784,0,0,"{'missing': {'bed': 0, 'bath': 0, 'city': 0, '..."
1,4,1,manual__2026-11-27T05:21:58.525299+00:00,2026-11-27 05:21:31.514747+00:00,94551,cleaned,"[acre_lot, bath, bed, brokered_by, city, house...",94551,0,0,"{'missing': {'bed': 0, 'bath': 0, 'city': 0, '..."


## Raw and clean counts

In [3]:
with db.connect() as conn:
    for table in ['raw.properties', 'clean.properties', 'audit.inference_events']:
        print(table, conn.execute(text(f'SELECT count(*) FROM {table}')).scalar())


raw.properties 168335
clean.properties 168335
audit.inference_events 9


## Decisions and model history

In [4]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM audit.batch_decisions ORDER BY batch_number'), conn))
    display(pd.read_sql_query(text('SELECT * FROM audit.training_events ORDER BY id'), conn))


,group_number,batch_number,created_at,valid_rows,pending_rows,decision,reason,model_version,candidate_mae,champion_mae,promotion,run_id,metrics,report_key
0,4,0,2026-11-27 05:18:17.506002+00:00,73784,73784,train,initial baseline,1,44278.238084,NaN,initial model,manual__2026-11-27T05:18:08.604029+00:00,"{'psi': {}, 'schema': {'columns': ['acre_lot',...",group-4/batch-0/report.json
1,4,1,2026-11-27 05:22:22.733857+00:00,94551,94551,train,"bed distribution, bath distribution, house_siz...",2,44933.593909,47261.429877,lower validation MAE,manual__2026-11-27T05:21:58.525299+00:00,"{'psi': {'bed': 1.1983245484975218, 'bath': 1....",group-4/batch-1/report.json


,id,group_number,batch_number,created_at,mlflow_run_id,model_version,candidate_mae,champion_mae,promoted,reason,train_rows
0,1,4,0,2026-11-27 05:19:08.144607+00:00,499efd7a0e4c4401ada03c58cdfd0e2b,1,44278.238084,NaN,True,initial model,44123
1,2,4,1,2026-11-27 05:23:07.169170+00:00,7450a0a40dbe430fa8f8a7b61cb1b7e6,2,44933.593909,47261.429877,True,lower validation MAE,100966


## Sample predictions

In [5]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT created_at, prediction, model_version, pod, status, error FROM audit.inference_events ORDER BY id DESC LIMIT 10'), conn))


,created_at,prediction,model_version,pod,status,error
0,2026-11-27 05:25:03.607130+00:00,241387.765625,2,property-api-bb86f9f5b-nskh4,success,None
1,2026-11-27 05:22:01.814019+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None
2,2026-11-27 05:21:57.831162+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None
3,2026-11-27 05:21:57.386294+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None
4,2026-11-27 05:21:57.061449+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None
5,2026-11-27 05:21:56.770232+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None
6,2026-11-27 05:21:56.448938+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None
7,2026-11-27 05:21:29.038495+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None
8,2026-11-27 05:21:27.547258+00:00,239949.437500,1,property-api-bb86f9f5b-nskh4,success,None


## Training data coverage

In [6]:
with db.connect() as conn:
    display(pd.read_sql_query(text('''
        WITH available AS (
            SELECT group_number, count(*) AS clean_rows
            FROM clean.properties GROUP BY group_number
        ), champions AS (
            SELECT DISTINCT ON (group_number) group_number, batch_number, model_version, (to_jsonb(t)->>'train_rows')::int AS train_rows
            FROM audit.training_events t WHERE promoted
            ORDER BY group_number, batch_number DESC
        )
        SELECT a.group_number, a.clean_rows, c.batch_number, c.model_version, c.train_rows,
               round(100.0 * c.train_rows / nullif(a.clean_rows, 0), 1) AS train_pct_of_current_clean
        FROM available a LEFT JOIN champions c USING (group_number)
    '''), conn))
db.dispose()


,group_number,clean_rows,batch_number,model_version,train_rows,train_pct_of_current_clean
0,4,168335,1,2,100966,60.0
